# 01 · Ingest & Clean — Texas SBIR/STTR awards

**Run this notebook first.** It loads the raw SBIR.gov award export, filters to Texas and the
last ~10 award years, standardizes types, builds a stable firm key, (optionally) geocodes to
county, and writes a single cleaned dataframe (`data/tx_sbir_clean.parquet`) that every analysis
notebook (A–G) reads.

### Methodology & sources (see `../METHODOLOGY.md`, `../SOURCES.md`)
- **Source:** SBIR/STTR Award Data, produced by the U.S. **Small Business Administration** (aggregates all 11 agencies). https://www.sbir.gov/awards
- **Filters:** `state == TX`; `award_year` within the last ~10 years.
- **Firm key:** dedup on `uei` when present, else normalized firm name.
- **County geocoding (S1):** Census/HUD crosswalk — wired in once approved (cell is gated below).
- **Validation anchor:** NC SBTDC report lists TX at #3 (247 Phase I / 160 Phase II / ~$241M / 5.6%).

In [ ]:
# --- Setup (Colab-friendly) ---
# In Colab, uncomment the next line the first time:
# !pip install -q pandas numpy pyarrow openpyxl matplotlib plotly requests
import os, re
import numpy as np
import pandas as pd

# Point DATA_DIR at the folder holding the raw award file.
# In Colab with Google Drive:
#   from google.colab import drive; drive.mount('/content/drive')
#   DATA_DIR = '/content/drive/MyDrive/tx_sbir'
DATA_DIR = os.environ.get('DATA_DIR', '../data')

# Name of the raw export you dropped in DATA_DIR (.csv or .xlsx):
RAW_FILE = os.environ.get('RAW_FILE', 'sbir_award_data.csv')
YEARS_BACK = 10
raw_path = os.path.join(DATA_DIR, RAW_FILE)
print('Looking for:', raw_path)

In [ ]:
# --- Load (handles .csv or .xlsx) ---
def load_awards(path):
    if not os.path.exists(path):
        raise FileNotFoundError(
            f'Award file not found at {path}. Put the SBIR.gov export in DATA_DIR '
            f'or set RAW_FILE / DATA_DIR. See ../data/README.md.')
    if path.lower().endswith(('.xlsx', '.xls')):
        return pd.read_excel(path)
    return pd.read_csv(path, low_memory=False)

df = load_awards(raw_path)
print(f'Loaded {len(df):,} rows, {df.shape[1]} columns')

# Normalize column names to snake_case so downstream code is stable regardless of the
# export's exact casing/spacing.
def snake(c):
    c = re.sub(r'[^0-9a-zA-Z]+', '_', str(c)).strip('_').lower()
    return c
df.columns = [snake(c) for c in df.columns]
print('Columns:', list(df.columns))

In [ ]:
# --- Column resolution: map the export's columns to the fields we rely on ---
# SBIR.gov exports have varied slightly over time; resolve defensively.
ALIASES = {
    'firm':        ['firm', 'company', 'firm_name', 'company_name'],
    'state':       ['state'],
    'city':        ['city'],
    'zip':         ['zip', 'zip_code', 'zipcode'],
    'agency':      ['agency'],
    'branch':      ['branch'],
    'phase':       ['phase'],
    'program':     ['program'],
    'award_year':  ['award_year', 'year'],
    'award_amount':['award_amount', 'amount', 'award_amount_'],
    'award_date':  ['proposal_award_date', 'award_date', 'award_start_date'],
    'end_date':    ['contract_end_date', 'end_date'],
    'uei':         ['uei'],
    'duns':        ['duns'],
    'employees':   ['number_employees', 'num_employees', 'employees'],
    'women_owned': ['women_owned', 'woman_owned'],
    'hubzone':     ['hubzone_owned', 'hubzone'],
    'disadvantaged':['socially_economically_disadvantaged', 'disadvantaged'],
    'topic':       ['topic_code', 'topic'],
    'keywords':    ['research_area_keywords', 'keywords'],
    'contract':    ['contract', 'contract_number'],
}
resolved = {}
for canon, opts in ALIASES.items():
    hit = next((o for o in opts if o in df.columns), None)
    resolved[canon] = hit
missing = [k for k, v in resolved.items() if v is None]
print('Resolved:', {k: v for k, v in resolved.items() if v})
if missing:
    print('WARNING - not found (some optional):', missing)

In [ ]:
# --- Filter to Texas + last 10 COMPLETE years, with record-count logging ---
# The export already appears TX-only, but we filter defensively so the notebook is
# correct for any SBIR.gov export. We also drop a partial trailing year (e.g. the
# current year with a handful of awards) so year-over-year charts aren't misleading.
def col(canon):
    return resolved.get(canon)

FOCUS_YEARS = 10
n0 = len(df)
state_col = col('state')
tx = df[df[state_col].astype(str).str.strip().str.upper().eq('TX')].copy()
print(f'TX filter: {n0:,} -> {len(tx):,} rows')

yr = col('award_year')
tx[yr] = pd.to_numeric(tx[yr], errors='coerce')
counts = tx.groupby(yr).size()
last = int(counts.index.max())
prev3_med = counts.loc[counts.index < last].tail(3).median()
# treat the final year as partial (and exclude it) if it has < 40% of the recent typical volume
end = last - 1 if counts.get(last, 0) < 0.4 * prev3_med else last
start = end - FOCUS_YEARS + 1
before = len(tx)
tx = tx[tx[yr].between(start, end)].copy()
print(f'Focus window [{start}-{end}] (dropped partial {last}: {int(counts.get(last,0))} awards): {before:,} -> {len(tx):,} rows')

In [ ]:
# --- Standardize types & derived fields ---
# Dollars
amt = col('award_amount')
tx['award_amount_num'] = (tx[amt].astype(str)
                          .str.replace(r'[^0-9.\-]', '', regex=True)
                          .replace('', np.nan).astype(float))

# Phase -> {Phase I, Phase II}
def norm_phase(p):
    s = str(p).upper()
    if 'II' in s or '2' in s: return 'Phase II'
    if 'I' in s or '1' in s:  return 'Phase I'
    return 'Other'
tx['phase_norm'] = tx[col('phase')].map(norm_phase)

# Program -> SBIR / STTR
tx['program_norm'] = tx[col('program')].astype(str).str.upper().str.extract(r'(SBIR|STTR)', expand=False)

# Dates
for c in ['award_date', 'end_date']:
    if col(c):
        tx[c + '_dt'] = pd.to_datetime(tx[col(c)], errors='coerce')

# Stable firm key: UEI if present else normalized name
def norm_name(n):
    s = re.sub(r'[^0-9a-z ]', ' ', str(n).lower())
    s = re.sub(r'\b(inc|llc|ltd|corp|corporation|company|co|the|lp|llp)\b', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()
tx['firm_name_norm'] = tx[col('firm')].map(norm_name)
uei = col('uei')
if uei:
    tx['firm_key'] = tx[uei].astype(str).str.strip().replace({'': np.nan, 'nan': np.nan})
    tx['firm_key'] = tx['firm_key'].fillna(tx['firm_name_norm'])
else:
    tx['firm_key'] = tx['firm_name_norm']
print('Rows:', len(tx), '| Unique firms:', tx['firm_key'].nunique())

In [ ]:
# --- County geocoding (S1) -- GATED until the crosswalk source is approved ---
# Approach once approved: map city/zip -> Texas county via the Census geocoder or the
# HUD-USPS ZIP->County crosswalk (see ../SOURCES.md, row S1). Left as a clearly marked
# TODO so we never silently invent geography.
ENABLE_COUNTY = False  # flip to True after S1 is approved and the crosswalk is in data/
if ENABLE_COUNTY:
    raise NotImplementedError('Wire in the approved Census/HUD county crosswalk here.')
else:
    tx['county'] = np.nan
    print('County geocoding skipped (source S1 not yet approved). Geography modules gated on this.')

In [ ]:
# --- Validation against the NC SBTDC anchor & basic sanity ---
summary = {
    'rows (awards)': len(tx),
    'unique firms': int(tx['firm_key'].nunique()),
    'Phase I awards': int((tx['phase_norm'] == 'Phase I').sum()),
    'Phase II awards': int((tx['phase_norm'] == 'Phase II').sum()),
    'total award $': float(tx['award_amount_num'].sum(skipna=True)),
    'year range': f"{int(tx[yr].min())}-{int(tx[yr].max())}",
}
for k, v in summary.items():
    print(f'{k:>18}: {v:,.0f}' if isinstance(v, float) else f'{k:>18}: {v}')
print('\nNote: NC SBTDC report lists TX at ~247 Phase I / 160 Phase II / ~$241M for its window;')
print('our numbers cover a different (10-yr) window, so expect larger totals -- same order of magnitude.')

In [ ]:
# --- Save the cleaned, analysis-ready dataframe ---
out = os.path.join(DATA_DIR, 'tx_sbir_clean.parquet')
tx.to_parquet(out, index=False)
print('Wrote', out, '|', len(tx), 'rows')
print('Downstream notebooks A-G load this file.')